# 04 — Evaluation and Report

Evaluate existing pretrained/Raw-trained/Adaptive-trained checkpoints on frozen Raw/Adaptive development inputs and inspect saved native results, per-class/condition comparisons, uncertainty and measured costs. This notebook never starts training or an upstream phase.

- **Inputs:** Existing native checkpoints or legacy six-class completion receipts, frozen protocol/policy, matched development exports, declared analysis plan and/or valid saved evaluation artifacts; loader measurements are optional.
- **Outputs:** `runs/development-v1/`: identity-checked native results/matches, tables/figures, analysis ledgers and report provenance.
- **Prerequisites:** Study environment and declared artifacts; explicit opt-in for missing inference or loader measurements. Final holdout stays locked.

**Outline**

1. [Configuration and Preflight](#configuration-preflight)
2. [Evaluation and Cache Reuse](#evaluation-cache)
3. [Per-class and Condition Analysis](#class-condition-analysis)
4. [Paired Comparisons and Uncertainty](#paired-uncertainty)
5. [Latency, Errors and Resource Costs](#latency-errors-resources)
6. [Report Export and Integrity Summary](#export-integrity)

Set paths in section 1, then **Restart & Run All**. Only this phase executes.
Missing or mismatched required artifacts fail; upstream phases are never launched.
The historical six-class taxonomy, including railway `train`, stays unchanged.

<a id="configuration-preflight"></a>

## 1. Configuration and Preflight

Default to saved evaluations and point estimates. Full resampling and new loader measurements are explicit choices; absent loader records are reported as pending. Set ALLOW_INFERENCE or MEASURE_LOADERS explicitly for missing phase-local work; these settings never start another phase.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from vehicle_detection.adaptive import validate_policy
from vehicle_detection.common import read_json, require_paths
from vehicle_detection.development_report import development_report
from vehicle_detection.evaluation import completed_checkpoints, evaluate_development
from vehicle_detection.loader_benchmark import benchmark_loader
from vehicle_detection.offline import verify_alignment
from vehicle_detection.protocol import validate_protocol

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
PROTOCOL_PATH = ROOT / "runs/data-v1/protocol.json"
POLICY_PATH = ROOT / "runs/preprocessing-v1/policy.json"
TRAIN_ROOT = ROOT / "runs/training-native-s10-v1"
PRETRAINED = ROOT / "weights/yolo26n.pt"
INPUTS = {
    "raw": ROOT / "runs/data-v1/datasets/raw",
    "adaptive": ROOT / "runs/preprocessing-v1/development/adaptive",
}
OUTPUT = ROOT / "runs/development-native-v1"
ALLOW_INFERENCE = False
MEASURE_LOADERS = False
BOOTSTRAP_RESAMPLES = 0  # Set 100 explicitly for the prospective uncertainty analysis.
PLAN = ROOT / "docs/development-v1/PLAN.md"
require_paths(
    plan=PLAN,
    protocol=PROTOCOL_PATH,
    policy=POLICY_PATH,
    pretrained=PRETRAINED,
    training=TRAIN_ROOT / "pair.json",
    **{k: v / "verified.json" for k, v in INPUTS.items()},
)
protocol = validate_protocol(read_json(PROTOCOL_PATH))
policy = validate_policy(read_json(POLICY_PATH), protocol)
checkpoints = {"pretrained": PRETRAINED, **completed_checkpoints(TRAIN_ROOT, protocol)}
verify_alignment(list(INPUTS.values()), deep=False)
if not ALLOW_INFERENCE:
    require_paths(
        **{
            f"saved_{m}_{t}": OUTPUT / f"{m}-{t}/native_result.json"
            for m in checkpoints
            for t in INPUTS
        }
    )

<a id="evaluation-cache"></a>

## 2. Evaluation and Cache Reuse

Reuse completed results only when checkpoint/data/settings/code and saved artifacts match. Changed identities require a new output root. Fresh native validation uses the correct COCO-to-six adapter or native six-class head.

In [ ]:
results = {}
for model_name, checkpoint in checkpoints.items():
    for input_name, dataset in INPUTS.items():
        name = f"{model_name}-{input_name}"
        results[name] = evaluate_development(
            protocol,
            policy,
            dataset,
            checkpoint,
            OUTPUT / name,
            pretrained=model_name == "pretrained",
            allow_inference=ALLOW_INFERENCE,
        )

<a id="class-condition-analysis"></a>

## 3. Per-class and Condition Analysis

Describe the full 3×2 development matrix and per-class AP. Missing/rare support remains visible; no class or image record is removed.

### 3.1 Overall and Per-class Metrics

In [ ]:
matrix = pd.DataFrame(
    [
        {"model": model, "input": treatment, **results[f"{model}-{treatment}"]["overall"]}
        for model in checkpoints
        for treatment in INPUTS
    ]
)
display(matrix)
per_class = pd.DataFrame(
    [{"evaluation": name, **row} for name, result in results.items() for row in result["classes"]]
)
display(per_class)
print("Native exported-file timing; excludes offline enhancement and application I/O:")
display(pd.DataFrame({name: r["speed_ms"] for name, r in results.items()}).T)

In [ ]:
matrix.pivot(index="model", columns="input", values="mAP50_95").plot.bar(
    figsize=(8, 3), rot=20, title="Native development mAP50:95 · six observed classes"
)
plt.tight_layout()
plt.show()

### 3.2 Natural Conditions and Support Counts

Read weather/time/scene slices at their natural prevalence. Reaggregate saved native matches without new inference. Zero bootstrap draws reports points only; set 100 explicitly for the prospective uncertainty analysis. Separate cache roots preserve each analysis identity.

In [ ]:
report = development_report(
    protocol,
    policy,
    checkpoints,
    OUTPUT,
    OUTPUT / f"analysis-b{BOOTSTRAP_RESAMPLES}",
    PLAN,
    resamples=BOOTSTRAP_RESAMPLES,
)
conditions = pd.DataFrame(report["conditions"])
display(conditions.drop(columns="classes"))

<a id="paired-uncertainty"></a>

## 4. Paired Comparisons and Uncertainty

Use the prospective completion plan while disclosing that S10 outcomes were already observed. When requested, paired source-group draws use a fixed GT-supported class denominator. With zero draws, comparisons are points only and uncertainty stays pending; unsupported slices/classes receive no invented CI.

In [ ]:
comparisons = pd.DataFrame(report["comparisons"])
display(comparisons.drop(columns=["draws", "per_class", "estimand"], errors="ignore"))
print(report["limitation"])
display(pd.DataFrame(report["errors"]))
print("Error ranking uses display confidence 0.25 / IoU0.50; metrics retain confidence 0.001.")
print("Crowded frames can dominate count rankings; native GT/prediction batch plots are above.")

overall = comparisons[comparisons.axis == "overall"]
if "recovery" in overall:
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, row in overall.iterrows():
        if row.status in {"exploratory", "not_resampled"}:
            if row.status == "exploratory":
                ax.plot([row.recovery_ci_low, row.recovery_ci_high], [i, i], color="black")
            ax.scatter(row.recovery, i)
    ax.set_yticks(overall.index, overall.before + " → " + overall.after)
    ax.axvline(0, color="gray")
    ax.set(title="Fixed supported-class AP delta · intervals only when explicitly computed")
    plt.tight_layout()
    plt.show()

Nominal intervals are exploratory, conditional on class presence and unadjusted for multiplicity. They describe development-group variability, not training-seed uncertainty or confirmed Adaptive superiority.

<a id="latency-errors-resources"></a>

## 5. Latency, Errors and Resource Costs

Keep native exported-file stage timing, offline preprocessing, isolated CPU loading and aggregate GPU training costs distinct. Inspect labeled/prediction batches and ranked error counts at their stated display threshold.

### 5.1 Native GT and Prediction Examples

In [ ]:
for name in results:
    display(Image(filename=str(OUTPUT / name / "confusion_matrix_normalized.png"), width=650))
    labels = sorted((OUTPUT / name).glob("val_batch*_labels.jpg"))
    if labels:
        display(Image(filename=str(labels[0]), width=900))
    examples = sorted((OUTPUT / name).glob("val_batch*_pred.jpg"))
    if examples:
        display(Image(filename=str(examples[0]), width=900))
print("No final-test execution or model-selection claims from this exploratory matrix.")

### 5.2 Isolated Loading, Training Costs and Observed Scale

In [ ]:
training_pair = read_json(TRAIN_ROOT / "pair.json")
selected_ids = set(training_pair["training"]["raw"]["ids"])
training_rows = [r for r in protocol["splits"]["training_pool"] if r["id"] in selected_ids]
loader_results = {}
for branch in ("raw", "adaptive"):
    measurement = OUTPUT / f"loader-{branch}/measurement.json"
    if MEASURE_LOADERS or measurement.is_file():
        loader_results[branch] = benchmark_loader(
            training_pair["training"][branch]["path"],
            training_rows,
            "raw_reencoded" if branch == "raw" else "adaptive",
            None if branch == "raw" else policy["recipes"],
            training_pair["settings"],
            OUTPUT / f"loader-{branch}",
            allow_measurement=MEASURE_LOADERS,
        )
    else:
        loader_results[branch] = {"status": "pending", "images_per_second": None, "workers": None}

display(
    pd.DataFrame(
        [
            {
                "branch": b,
                "status": r.get("status", "measured"),
                "loader_images_per_second": r["images_per_second"],
                "workers": r["workers"],
            }
            for b, r in loader_results.items()
        ]
    )
)
resource_rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for branch in ("raw", "adaptive"):
    history = pd.read_csv(TRAIN_ROOT / branch / "results.csv")
    history.plot(x="epoch", y="metrics/mAP50-95(B)", ax=axes[0], label=branch)
    history.plot(x="epoch", y="train/box_loss", ax=axes[1], label=branch)
    # Native results.csv measures elapsed run time when the pinned build exposes it.
    # Historical custom per-epoch receipts are optional and never required here.
    elapsed = float(history["time"].iloc[-1]) if "time" in history and len(history) else None
    resource_rows.append(
        {
            "branch": branch,
            "training_images": len(training_rows),
            "training_groups": len({r["group"] for r in training_rows}),
            "epochs": len(history),
            "native_elapsed_seconds": elapsed,
            "images_per_elapsed_second": len(training_rows) * len(history) / elapsed if elapsed and elapsed > 0 else None,
        }
    )
axes[0].set_title("Original shared Raw development selection")
axes[1].set_title("Training box loss")
plt.tight_layout()
plt.show()
display(pd.DataFrame(resource_rows))
print("Only S10 has measured scale outcomes; S20/S40 are unexecuted, never interpolated.")
fig, ax = plt.subplots(figsize=(7, 3))
fraction = len(training_rows) / len(protocol["splits"]["training_pool"])
for branch in ("raw", "adaptive"):
    ap = results[f"{branch}-raw"]["overall"]["mAP50_95"]
    ax.scatter(fraction, ap, label=branch)
ax.set(
    xlabel="Fraction of training pool",
    ylabel="Raw-development mAP50:95",
    title="Observed scale point: S10 only",
    xlim=(0, 1),
)
ax.legend()
plt.tight_layout()
plt.show()

Only S10 has observed scale outcomes. Sequential warm-cache loader results do not establish a causal speed advantage, and a macro AP point alone does not justify expansion.

<a id="export-integrity"></a>

## 6. Report Export and Integrity Summary

Export derived tables and provenance. Model/scale selection and final release remain pending scientific work; report execution does not lock new model choices or open holdout.

In [ ]:
from vehicle_detection.common import digest, freeze_json

REPORT = OUTPUT / f"analysis-b{BOOTSTRAP_RESAMPLES}"
matrix.to_csv(REPORT / "development_matrix.csv", index=False)
per_class.to_csv(REPORT / "per_class.csv", index=False)
conditions.drop(columns="classes").to_csv(REPORT / "conditions.csv", index=False)
freeze_json(
    REPORT / "report_provenance.json",
    {
        "protocol_id": protocol["manifest_id"],
        "policy_id": policy["policy_id"],
        "analysis_sha256": digest(report),
        "final_test_released": False,
    },
)
print("Report artifacts:", REPORT)
print("Checkpoint/scale selection remains separate; final-test pixels remain untouched.")